In [1]:
import cv2
from ultralytics import YOLO
import os
import time
import numpy as np
import faiss
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

In [6]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

import clip_encoder

# Load YOLO
model = YOLO("yolov8n.pt")

# ---- Load or create FAISS index ----
d = 512  # CLIP ViT-B/32 embedding dimension

index_path = "data/index.faiss"
paths_path = "data/paths.txt"

if os.path.exists(index_path) and os.path.exists(paths_path):
    print("[INFO] Loading existing FAISS index...")
    index = faiss.read_index(index_path)
    with open(paths_path, "r") as f:
        gallery_paths = f.read().splitlines()
else:
    print("[INFO] No FAISS index found, creating a new one...")
    index = faiss.IndexFlatIP(d)  # cosine similarity after normalize
    gallery_paths = []

# Output directory for live crops
output_dir = "data/cropped_objects"
os.makedirs(output_dir, exist_ok=True)

# File where website drops new queries
new_queries_file = "data/new_queries.txt"

# Similarity threshold
SIM_THRESHOLD = 0.6

[INFO] No FAISS index found, creating a new one...


In [7]:
cap = cv2.VideoCapture(0)
frame_count = 0

print("Camera running... press 'q' to quit.")

def embed_text(text: str):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True)
    with torch.no_grad():
        emb = clip_model.get_text_features(**inputs).cpu().numpy()
    return emb

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]

    # ---- Step 1: Detect objects ----
    for i, box in enumerate(results.boxes.xyxy):
        x1, y1, x2, y2 = map(int, box[:4])
        cropped = frame[y1:y2, x1:x2]

        # Save crop (optional)
        timestamp = int(time.time() * 1000)
        filename = f"{output_dir}/crop_{frame_count}_{i}_{timestamp}.jpg"
        cv2.imwrite(filename, cropped)

        # Embed crop
        emb = clip_encoder.embed_image(filename).astype("float32")
        faiss.normalize_L2(emb)

        # Search in FAISS index
        D, I = index.search(emb, k=1)
        best_score = D[0][0]
        best_match_idx = I[0][0]

        if best_score >= SIM_THRESHOLD:
            matched_path = gallery_paths[best_match_idx]
            text = f"FOUND: {os.path.basename(matched_path)} ({best_score:.2f})"
            color = (0, 0, 255)
        else:
            text = f"No match ({best_score:.2f})"
            color = (0, 255, 0)

        # Draw box + label
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # ---- Step 2: Check for new queries from website ----
    if os.path.exists(new_queries_file):
        with open(new_queries_file, "r") as f:
            new_queries = f.read().splitlines()

        if new_queries:
            for q in new_queries:
                if q.startswith("img:"):
                    q_path = q.replace("img:", "").strip()
                    if os.path.exists(q_path):
                        print(f"Adding new IMAGE query: {q_path}")
                        emb = clip_encoder.embed_image(q_path).astype("float32")
                        faiss.normalize_L2(emb)
                        index.add(emb)
                        gallery_paths.append(q_path)

                elif q.startswith("txt:"):
                    q_text = q.replace("txt:", "").strip()
                    if q_text:
                        print(f"Adding new TEXT query: '{q_text}'")
                        emb = embed_text(q_text).astype("float32")
                        faiss.normalize_L2(emb)
                        index.add(emb)
                        gallery_paths.append(f"TEXT:{q_text}")

            # Save updated index + gallery
            faiss.write_index(index, "data/index.faiss")
            with open("data/paths.txt", "w") as f:
                f.write("\n".join(gallery_paths))

            # Clear new queries file
            open(new_queries_file, "w").close()

    # ---- Step 3: Show live feed ----
    cv2.imshow("Live-Feed", frame)
    frame_count += 1

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Camera running... press 'q' to quit.

0: 480x640 1 person, 11.0ms
Speed: 2.6ms preprocess, 11.0ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 9.8ms
Speed: 1.4ms preprocess, 9.8ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 14.5ms
Speed: 2.3ms preprocess, 14.5ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 23.3ms
Speed: 2.8ms preprocess, 23.3ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 22.4ms
Speed: 2.8ms preprocess, 22.4ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 22.2ms
Speed: 2.9ms preprocess, 22.2ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 41.0ms
Speed: 3.7ms preprocess, 41.0ms inference, 4.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 46.9ms
Speed: 4.2ms preprocess, 46.9ms inference, 2.0ms postp